In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
# --- CONFIGURAÇÃO PARA RODAR LOCAL (VS CODE) ---
# Desativa WandB para não pedir login
os.environ["WANDB_DISABLED"] = "true"

# Verifica se tem GPU (Cuda para Nvidia ou MPS para Mac M1/M2)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"🔥 Rodando em: {device.upper()}")

In [ ]:
# ==========================================
# 1. CARREGAMENTO E LIMPEZA
# ==========================================
print("📂 Carregando dados...")
try:
    df = pd.read_json("merged.json", lines=True)
except ValueError:
    df = pd.read_json("merged.json")

# Limpeza e garantia de listas
df = df[['passo', 'utensilios_equipamentos']]
df.columns = ['text', 'labels']
df['labels'] = df['labels'].apply(lambda x: x if isinstance(x, list) else [])

In [ ]:
# ==========================================
# 2. PREPARAÇÃO DOS LABELS
# ==========================================
mlb = MultiLabelBinarizer()
labels_matrix = mlb.fit_transform(df['labels'])
label_list = mlb.classes_.tolist()
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(f"✅ Utensílios únicos: {len(label_list)}")

# Divisão
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), labels_matrix, test_size=0.1, random_state=42
)

In [ ]:
# ==========================================
# 3. TOKENIZAÇÃO INTELIGENTE
# ==========================================
print("⚙️ Tokenizando...")
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Dataset Otimizado para PyTorch
class UtensilDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = UtensilDataset(train_texts, train_labels, tokenizer)
val_dataset = UtensilDataset(val_texts, val_labels, tokenizer)

In [ ]:
# ==========================================
# 4. MODELO & CONGELAMENTO (O PULO DO GATO)
# ==========================================
print("🧠 Carregando Modelo...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    problem_type="multi_label_classification",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# --- OTIMIZAÇÃO MAXIMA: FREEZING ---
# Congela o BERT base. Treina só o classificador final.
# Isso reduz o uso de memória drasticamente e acelera o treino.
for param in model.bert.parameters():
    param.requires_grad = False

print("❄️  Camadas do BERT congeladas. Treinando apenas o classificador.")

In [ ]:
# ==========================================
# 5. TREINAMENTO
# ==========================================
batch_size = 8  # Pequeno para caber na memória
grad_accum = 4  # Acumula 4 passos (Simula batch de 32!)

training_args = TrainingArguments(
    output_dir="./resultados_local",
    num_train_epochs=5,              # 5 Épocas é rapidinho com freezing
    
    # Configuração de Memória/Velocidade
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    
    # Trabalhadores de CPU (Carrega dados enquanto GPU trabalha)
    dataloader_num_workers=2,        # Ajuste conforme seus núcleos de CPU
    
    # Precisão Mista (Se tiver GPU Nvidia)
    fp16=True if device == "cuda" else False, 
    
    # Logs e Salvamento
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,              # Guarda só o último modelo para não encher o disco
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("🚀 Iniciando Treinamento...")
trainer.train()

# Salvar o modelo final para usar depois
trainer.save_model("./modelo_final_utensilios")
tokenizer.save_pretrained("./modelo_final_utensilios")
print("💾 Modelo salvo em './modelo_final_utensilios'")

In [ ]:
# ==========================================
# 6. TESTE RÁPIDO
# ==========================================
def prever(texto):
    inputs = tokenizer(texto, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits).cpu().numpy()[0]
    
    # Filtro: Mostra apenas o que tem > 50% de chance
    results = {id2label[i]: float(p) for i, p in enumerate(probs) if p > 0.5}
    return results

print("\n--- Teste Final ---")
teste1 = "Bata as claras em neve na batedeira e misture com uma colher."
print(f"Frase: {teste1}")
print(f"Resultado: {prever(teste1)}")